# 4.4 DPO / DPO

从 4.3 的闭式最优解出发，把奖励模型和强化学习合并成一个分类损失。

Starting from the closed-form optimum of 4.3, merge the reward model and reinforcement learning into a single classification loss.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 实现 DPO 损失 / Implement the DPO loss

实现整段回答的对数概率、DPO 损失和隐含奖励。测试会检查五条性质，包括策略等于参考模型时损失恰好是 $\ln 2$，以及被排错的回答对得到更大的梯度。

Implement the log-probability of a whole response, the DPO loss and the implicit reward. The tests check five properties, including that the loss is exactly $\ln 2$ when the policy equals the reference model, and that a wrongly ranked pair receives the larger gradient.

In [ ]:
import math
import torch
import torch.nn.functional as F


def sequence_logprob(logits, tokens, mask):
    """log pi(y | x) of a whole response: the sum of the log-probabilities of its tokens.

    logits: (B, T, V) where logits[:, t] is the prediction FOR tokens[:, t]  (already aligned)
    tokens: (B, T) long.  mask: (B, T) bool, True on the response tokens that count.
    Returns (B,).
    """
    # TODO 1: log-softmax, pick the token at every position, zero out masked positions, sum over T
    raise NotImplementedError


def dpo_loss(pi_logp_w, pi_logp_l, ref_logp_w, ref_logp_l, beta=0.1):
    """Direct Preference Optimization loss, averaged over the batch.

    pi_logp_w / pi_logp_l:   log pi_theta(y_w | x) and log pi_theta(y_l | x), shape (B,)
    ref_logp_w / ref_logp_l: the same for the frozen reference model
    loss = -log sigmoid( beta * [ (pi_w - ref_w) - (pi_l - ref_l) ] )
    """
    # TODO 2
    raise NotImplementedError


def implicit_reward(pi_logp, ref_logp, beta=0.1):
    """The reward that DPO implicitly assigns to a response:  beta * log( pi_theta / pi_ref )."""
    # TODO 3
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math
import torch
import torch.nn.functional as F

torch.manual_seed(0)
B, T, V = 3, 6, 11
logits = torch.randn(B, T, V, dtype=torch.float64)
tokens = torch.randint(0, V, (B, T))
mask = torch.tensor([[0, 0, 1, 1, 1, 1], [0, 1, 1, 1, 0, 0], [0, 0, 0, 1, 1, 0]], dtype=torch.bool)
lp = sequence_logprob(logits, tokens, mask)
assert lp.shape == (B,) and (lp < 0).all()
ref = torch.stack([sum(torch.log_softmax(logits[b, t], -1)[tokens[b, t]] for t in range(T) if mask[b, t]) for b in range(B)])
assert torch.allclose(lp, ref, atol=1e-10), "sum the log-probabilities of the unmasked tokens"

pw, pl = torch.tensor([-5.0, -7.0], dtype=torch.float64), torch.tensor([-6.0, -4.0], dtype=torch.float64)
rw, rl = torch.tensor([-5.5, -6.0], dtype=torch.float64), torch.tensor([-5.0, -5.0], dtype=torch.float64)
u = 0.5 * ((pw - rw) - (pl - rl))
assert torch.allclose(dpo_loss(pw, pl, rw, rl, beta=0.5), -torch.log(torch.sigmoid(u)).mean(), atol=1e-12)

# 1. before any training the policy equals the reference, and the loss is exactly ln 2
assert abs(dpo_loss(rw, rl, rw, rl).item() - math.log(2)) < 1e-12
# 2. raising the winner or lowering the loser reduces the loss
base = dpo_loss(pw, pl, rw, rl, beta=0.5)
assert dpo_loss(pw + 1, pl, rw, rl, beta=0.5) < base and dpo_loss(pw, pl - 1, rw, rl, beta=0.5) < base
# 3. only the difference to the reference matters: a constant shift of both responses changes nothing
assert torch.allclose(dpo_loss(pw + 3, pl + 3, rw, rl, beta=0.5), base, atol=1e-12)
# 4. the gradient pushes the two log-probabilities apart, harder when the model ranks the pair wrongly
a = torch.tensor([-5.0, -5.0], dtype=torch.float64, requires_grad=True)       # log pi(y_w)
b = torch.tensor([-9.0, -1.0], dtype=torch.float64, requires_grad=True)       # log pi(y_l): pair 0 ranked right, pair 1 wrong
z = torch.zeros(2, dtype=torch.float64)
dpo_loss(a, b, z, z, beta=1.0).backward()
assert (a.grad < 0).all() and (b.grad > 0).all(), "descent raises log pi(y_w) and lowers log pi(y_l)"
assert a.grad[1].abs() > 10 * a.grad[0].abs(), "the wrongly ranked pair gets the larger gradient"

assert torch.allclose(implicit_reward(pw, rw, 0.5) - implicit_reward(pl, rl, 0.5), u)
print("m44_dpo_loss: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

整段回答的对数概率是各个 token 对数概率之和。先用 `gather` 取出每个位置上目标 token 的对数概率，乘以掩码后沿序列维度求和。

The log-probability of a whole response is the sum of the log-probabilities of its tokens. Pick the log-probability of the target token at every position with `gather`, multiply by the mask and sum along the sequence dimension.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

损失是 `-F.logsigmoid(beta * ((pi_w - ref_w) - (pi_l - ref_l))).mean()`。

The loss is `-F.logsigmoid(beta * ((pi_w - ref_w) - (pi_l - ref_l))).mean()`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch
import torch.nn.functional as F


def sequence_logprob(logits, tokens, mask):
    logp = torch.log_softmax(logits, dim=-1).gather(-1, tokens.unsqueeze(-1)).squeeze(-1)   # (B, T)
    return (logp * mask).sum(dim=-1)


def dpo_loss(pi_logp_w, pi_logp_l, ref_logp_w, ref_logp_l, beta=0.1):
    u = beta * ((pi_logp_w - ref_logp_w) - (pi_logp_l - ref_logp_l))
    return -F.logsigmoid(u).mean()


def implicit_reward(pi_logp, ref_logp, beta=0.1):
    return beta * (pi_logp - ref_logp)
```

</details>

## 2. 在玩具策略上验证 DPO 收敛到 RLHF 的最优解 / Verify on a toy policy that DPO reaches the RLHF optimum

沿用 4.3 的玩具问题，一个提示、6 个候选回答。模拟的标注者按一个隐藏的奖励做两两比较。只用这些比较结果和 DPO 损失训练策略，全程不使用奖励的数值。测试会检查训练得到的策略与 4.3 推出的闭式最优解一致，并且隐含奖励还原了隐藏的奖励。

Continue with the toy problem of 4.3, one prompt and 6 candidate responses. Simulated annotators compare pairs according to a hidden reward. Train the policy with these comparisons and the DPO loss alone, never using the reward values. The tests check that the trained policy agrees with the closed-form optimum derived in 4.3 and that the implicit reward recovers the hidden one.

In [ ]:
import torch
import torch.nn.functional as F

# A toy version of the whole DPO setting. There is one prompt and K possible responses.
# The policy is pi = softmax(theta), the reference is fixed, and annotators compare pairs of
# responses according to a hidden reward (Bradley-Terry model, as in module 4.3).


def sample_preferences(rewards, n_pairs, generator=None):
    """Simulated annotation. Returns (winners, losers), two long tensors of response indices.

    Draw two different responses i, j uniformly; i wins with probability sigmoid(r_i - r_j).
    Nothing to change here.
    """
    K = len(rewards)
    i = torch.randint(0, K, (n_pairs,), generator=generator)
    j = torch.randint(0, K, (n_pairs,), generator=generator)
    keep = i != j
    i, j = i[keep], j[keep]
    i_wins = torch.rand(len(i), generator=generator, dtype=rewards.dtype) < torch.sigmoid(rewards[i] - rewards[j])
    return torch.where(i_wins, i, j), torch.where(i_wins, j, i)


def dpo_train(logp_ref, winners, losers, beta, steps=2000, lr=2.0):
    """Minimise the DPO loss by plain gradient descent, starting from the reference policy.

    theta starts at logp_ref. Each step:
      logp = log_softmax(theta)
      u    = beta * [ (logp[w] - logp_ref[w]) - (logp[l] - logp_ref[l]) ]
      loss = mean( -logsigmoid(u) )
    Returns the final policy as probabilities (K,).
    """
    # TODO 1
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch

torch.manual_seed(0)
K, beta = 6, 0.5
logp_ref = torch.log_softmax(torch.randn(K, dtype=torch.float64), -1)
r = torch.randn(K, dtype=torch.float64)                              # the hidden reward the annotators follow

g = torch.Generator().manual_seed(0)
w, l = sample_preferences(r, 20000, g)
pi = dpo_train(logp_ref, w, l, beta)
assert pi.shape == (K,) and abs(pi.sum().item() - 1) < 1e-9

# The closed-form optimum of RLHF from module 4.3: pi* proportional to pi_ref * exp(r / beta)
star = torch.softmax(logp_ref + r / beta, -1)
err = (pi - star).abs().max().item()
assert err < 0.03, f"DPO should land on the RLHF optimum without ever seeing the reward, max error {err:.3f}"

# The implicit reward beta * log(pi / pi_ref) recovers the hidden reward up to a constant.
implied = beta * (pi.log() - logp_ref)
diff = (implied - implied.mean()) - (r - r.mean())
assert diff.abs().max() < 0.15, "the implicit reward should match the hidden one up to a shift"

# With no data the policy stays where it started.
none = torch.tensor([], dtype=torch.long)
assert torch.allclose(dpo_train(logp_ref, w[:1], l[:1], beta, steps=0), logp_ref.exp())
print(f"m44_dpo_toy: all tests passed (max error to the RLHF optimum {err:.4f})")
print("  reference :", [round(v, 3) for v in logp_ref.exp().tolist()])
print("  DPO result:", [round(v, 3) for v in pi.tolist()])
print("  RLHF optimum:", [round(v, 3) for v in star.tolist()])

<details><summary><b>提示 1 / Hint 1</b></summary>

每一步先算 `logp = torch.log_softmax(theta, -1)`，再用 `logp[winners]` 和 `logp[losers]` 取出每次比较中两个回答的对数概率。

Each step first computes `logp = torch.log_softmax(theta, -1)` and then picks the log-probabilities of the two responses of each comparison with `logp[winners]` and `logp[losers]`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

手动梯度下降的写法与 4.3 的奖励模型练习相同，更新放在 `torch.no_grad()` 里，每步之后把梯度清零。

Manual gradient descent is written as in the reward-model exercise of 4.3, with the update inside `torch.no_grad()` and the gradient zeroed after every step.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
import torch.nn.functional as F


def sample_preferences(rewards, n_pairs, generator=None):
    K = len(rewards)
    i = torch.randint(0, K, (n_pairs,), generator=generator)
    j = torch.randint(0, K, (n_pairs,), generator=generator)
    keep = i != j
    i, j = i[keep], j[keep]
    i_wins = torch.rand(len(i), generator=generator, dtype=rewards.dtype) < torch.sigmoid(rewards[i] - rewards[j])
    return torch.where(i_wins, i, j), torch.where(i_wins, j, i)


def dpo_train(logp_ref, winners, losers, beta, steps=2000, lr=2.0):
    theta = logp_ref.clone().requires_grad_(True)
    for _ in range(steps):
        logp = torch.log_softmax(theta, dim=-1)
        u = beta * ((logp[winners] - logp_ref[winners]) - (logp[losers] - logp_ref[losers]))
        loss = -F.logsigmoid(u).mean()
        loss.backward()
        with torch.no_grad():
            theta -= lr * theta.grad
            theta.grad.zero_()
    return torch.softmax(theta.detach(), dim=-1)
```

</details>

## 3. 从 Bradley-Terry 模型推导 DPO 损失 / Derive the DPO loss from the Bradley-Terry model

这是大模型岗位面试中的常见推导题。合上上面的推导，在纸上完成。
(a) 已知 $\pi^{*}(y \mid x) = \pi_{\text{ref}}(y \mid x) \exp(r(x, y)/\beta) / Z(x)$。解出 $r(x, y)$。
(b) 把 (a) 的结果代入 $P(y_w \succ y_l \mid x) = \sigma(r(x, y_w) - r(x, y_l))$，说明 $Z(x)$ 为什么消失，以及这一点为什么重要。
(c) 写出 DPO 损失。
(d) 证明策略等于参考模型时损失为 $\ln 2$。
(e) 写出损失对 $\theta$ 的梯度，并解释权重 $\sigma(-u)$ 的作用。

This is a common derivation in interviews for LLM roles. Close the derivation above and work on paper.
(a) Given $\pi^{*}(y \mid x) = \pi_{\text{ref}}(y \mid x) \exp(r(x, y)/\beta) / Z(x)$, solve for $r(x, y)$.
(b) Substitute the result of (a) into $P(y_w \succ y_l \mid x) = \sigma(r(x, y_w) - r(x, y_l))$, and say why $Z(x)$ disappears and why this matters.
(c) Write the DPO loss.
(d) Show that the loss is $\ln 2$ when the policy equals the reference model.
(e) Write the gradient of the loss with respect to $\theta$ and explain the role of the weight $\sigma(-u)$.

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 两边取对数。

(a) Take logs on both sides.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(e) $\dfrac{d}{du}\big[-\log\sigma(u)\big] = -\sigma(-u)$。

(e) $\dfrac{d}{du}\big[-\log\sigma(u)\big] = -\sigma(-u)$.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) $\log \pi^{*} = \log \pi_{\text{ref}} + r/\beta - \log Z$，所以 $r(x, y) = \beta \log \dfrac{\pi^{*}(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)$。

(b) $r(x, y_w) - r(x, y_l) = \beta \log \dfrac{\pi^{*}(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \dfrac{\pi^{*}(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}$。两个回答对应同一个提示 $x$，$\beta \log Z(x)$ 在相减时抵消。$Z(x)$ 需要对所有可能的回答求和，无法计算，它的消失使偏好概率可以只用策略和参考模型的对数概率表示。

(c) $\mathcal{L}_{\text{DPO}} = -\mathbb{E}\Big[\log \sigma\Big(\beta \log \dfrac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \dfrac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\Big)\Big]$。

(d) $\pi_\theta = \pi_{\text{ref}}$ 时两个对数比都是 0，$\sigma(0) = \tfrac12$，损失为 $-\log \tfrac12 = \ln 2$。

(e) 记括号内为 $u$，则 $\nabla_\theta \mathcal{L} = -\beta\, \mathbb{E}\big[\sigma(-u)\big(\nabla_\theta \log \pi_\theta(y_w \mid x) - \nabla_\theta \log \pi_\theta(y_l \mid x)\big)\big]$。$\sigma(-u)$ 在隐含奖励把这一对排错时接近 1，在排对且差距大时接近 0，所以更新主要集中在模型还没有排对的样本上。

---

(a) $\log \pi^{*} = \log \pi_{\text{ref}} + r/\beta - \log Z$, so $r(x, y) = \beta \log \dfrac{\pi^{*}(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)$.

(b) $r(x, y_w) - r(x, y_l) = \beta \log \dfrac{\pi^{*}(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \dfrac{\pi^{*}(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}$. Both responses belong to the same prompt $x$, so $\beta \log Z(x)$ cancels in the subtraction. $Z(x)$ requires a sum over all possible responses and cannot be computed, and its disappearance lets the preference probability be expressed with log-probabilities of the policy and the reference model alone.

(c) $\mathcal{L}_{\text{DPO}} = -\mathbb{E}\Big[\log \sigma\Big(\beta \log \dfrac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \dfrac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\Big)\Big]$.

(d) With $\pi_\theta = \pi_{\text{ref}}$ both log-ratios are 0, $\sigma(0) = \tfrac12$ and the loss is $-\log \tfrac12 = \ln 2$.

(e) Calling the argument $u$, $\nabla_\theta \mathcal{L} = -\beta\, \mathbb{E}\big[\sigma(-u)\big(\nabla_\theta \log \pi_\theta(y_w \mid x) - \nabla_\theta \log \pi_\theta(y_l \mid x)\big)\big]$. $\sigma(-u)$ is near 1 when the implicit reward ranks the pair wrongly and near 0 when it ranks it correctly by a wide margin, so updates concentrate on the examples the model has not yet ranked correctly.

</details>